# Data Transformation and Normalization — Google Trends Dataset

Input: `trends_cleaned.csv` (output of the Data Cleaning assignment).
Output: `trends_transformed.csv`.

Steps: import -> derived columns -> dtype conversion -> categorical standardization ->
unit standardization -> Min-Max normalization -> original vs normalized comparison -> verification.

## 1. Import the cleaned dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("trends_cleaned.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Derived columns

| Column | Meaning |
|---|---|
| `rank_score` | 6 - rank_position, so higher = more popular (rank 1 -> 5 points) |
| `years_since_start` | year - first year in the dataset |
| `decade` | decade the trend belongs to |
| `query_length` | characters in the search query |
| `query_word_count` | words in the search query |
| `is_top_trend` | True when rank_position == 1 |
| `location_trend_count` | number of rows for that location |

In [ ]:
df["rank_score"] = 6 - df["rank_position"]
df["years_since_start"] = df["year"] - df["year"].min()
df["decade"] = (df["year"] // 10) * 10
df["query_length"] = df["query"].str.len()
df["query_word_count"] = df["query"].str.split().str.len()
df["is_top_trend"] = df["rank_position"].eq(1)
df["location_trend_count"] = df.groupby("location")["location"].transform("size")

df[["location", "year", "rank_position", "rank_score", "years_since_start",
    "decade", "query", "query_length", "query_word_count",
    "is_top_trend", "location_trend_count"]].head()

## 3. Convert columns to suitable data formats

In [ ]:
# Numeric columns -> integer types
num_cols = ["year", "rank_position", "rank_score", "years_since_start",
            "decade", "query_length", "query_word_count", "location_trend_count"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype("int64")

# Boolean columns -> real bools
for c in ["is_global", "is_top_trend"]:
    df[c] = df[c].astype(str).str.strip().str.lower().map(
        {"true": True, "false": False, "1": True, "0": False}).astype("bool")

# Low-cardinality text -> category dtype
for c in ["location", "category_standard"]:
    df[c] = df[c].astype("category")

df.dtypes

## 4. Standardize inconsistent categorical values

Collapse whitespace, apply a single casing rule, and unify equivalent spellings
(e.g. `US` / `U.S.A.` -> `United States`).

In [ ]:
def clean_text(s):
    # collapse repeated whitespace and trim
    return s.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

df["location"] = clean_text(df["location"]).str.title()
df["category"] = clean_text(df["category"]).str.title()
df["category_standard"] = clean_text(df["category_standard"]).str.title()
df["query"] = clean_text(df["query"])

location_aliases = {
    "Usa": "United States", "Us": "United States", "U.S.A.": "United States",
    "U.S.": "United States", "Uk": "United Kingdom", "U.K.": "United Kingdom",
    "Uae": "United Arab Emirates", "Global": "Worldwide", "World": "Worldwide",
    "Korea": "South Korea", "Republic Of Korea": "South Korea",
    "Russian Federation": "Russia", "Viet Nam": "Vietnam",
    "Czechia": "Czech Republic", "Holland": "Netherlands",
}
df["location"] = df["location"].replace(location_aliases).astype("category")

print("Locations after standardization:", df["location"].nunique())
sorted(df["location"].unique())[:15]

In [ ]:
df["category_standard"] = df["category_standard"].astype("category")
df["category_standard"].value_counts()

## 5. Standardize inconsistent units

* `year` — forced to a single 4-digit calendar-year unit; any 2-digit year (e.g. `08`) becomes `2008`.
* `rank_position` — kept on one 1..5 scale; the same information is mirrored as `rank_score`
  on a 5-point scale where higher = better.

In [ ]:
# Any 2-digit year -> 4-digit calendar year
two_digit = df["year"] < 100
df.loc[two_digit, "year"] = df.loc[two_digit, "year"] + 2000
print("Fixed 2-digit years:", int(two_digit.sum()))

print("Year range:", df["year"].min(), "-", df["year"].max())
print("Rank range:", df["rank_position"].min(), "-", df["rank_position"].max())
print("Rank score range:", df["rank_score"].min(), "-", df["rank_score"].max())

## 6. Min-Max Normalization

$$x_{norm} = \\frac{x - x_{min}}{x_{max} - x_{min}}$$

Applied to the numerical columns, producing `*_norm` columns scaled to [0, 1].

In [ ]:
def min_max(s):
    lo, hi = s.min(), s.max()
    if hi == lo:            # constant column -> all zeros, avoids divide-by-zero
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

norm_cols = ["year", "rank_position", "rank_score",
             "query_length", "query_word_count", "location_trend_count"]

for c in norm_cols:
    df[c + "_norm"] = min_max(df[c]).round(6)

df[[c + "_norm" for c in norm_cols]].describe()

## 7. Compare original vs normalized values

In [ ]:
comparison = pd.DataFrame({
    "column": norm_cols,
    "original_min": [df[c].min() for c in norm_cols],
    "original_max": [df[c].max() for c in norm_cols],
    "original_mean": [round(df[c].mean(), 2) for c in norm_cols],
    "norm_min": [df[c + "_norm"].min() for c in norm_cols],
    "norm_max": [df[c + "_norm"].max() for c in norm_cols],
    "norm_mean": [round(df[c + "_norm"].mean(), 4) for c in norm_cols],
})
comparison

In [ ]:
# Row-level view of the same values before and after scaling
pairs = [c for col in norm_cols for c in (col, col + "_norm")]
df[pairs].head(10)

In [ ]:
# Sanity check: every normalized column must sit inside [0, 1]
for c in norm_cols:
    assert df[c + "_norm"].between(0, 1).all(), f"{c}_norm out of range"
print("All normalized columns are within [0, 1].")

## 8. Verify the transformed dataset

In [ ]:
print("Shape:", df.shape)
missing = df.isna().sum()
print("\nMissing values:", missing[missing > 0].to_dict() or "None")
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate (location, year, category, rank_position):",
      df.duplicated(subset=["location", "year", "category", "rank_position"]).sum())

In [ ]:
checks = {
    "rank_position in 1..5": bool(df["rank_position"].between(1, 5).all()),
    "rank_score = 6 - rank_position": bool((df["rank_score"] == 6 - df["rank_position"]).all()),
    "year in 2001..2025": bool(df["year"].between(2001, 2025).all()),
    "query_length > 0": bool((df["query_length"] > 0).all()),
    "no leading/trailing spaces in query": bool((df["query"] == df["query"].str.strip()).all()),
    "is_top_trend matches rank 1": bool((df["is_top_trend"] == df["rank_position"].eq(1)).all()),
}
for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", name)
assert all(checks.values()), "Consistency check failed" 

## 9. Save the transformed dataset

In [ ]:
df.to_csv("trends_transformed.csv", index=False)
print("Saved trends_transformed.csv", df.shape)
df.head()

## Summary of transformation and normalization

1. **Import** — loaded `trends_cleaned.csv` with Pandas.
2. **Derived columns** — added `rank_score`, `years_since_start`, `decade`, `query_length`,
   `query_word_count`, `is_top_trend`, `location_trend_count`.
3. **Data formats** — numeric columns cast to `int64`, flags to `bool`, low-cardinality
   text columns (`location`, `category_standard`) to `category`.
4. **Categorical standardization** — whitespace collapsed, Title Case applied, and country
   aliases (`US`, `U.S.A.`, `UK`, `Global`, ...) mapped to one canonical name each.
5. **Unit standardization** — years forced to a 4-digit calendar-year unit; rank kept on a
   single 1..5 scale and mirrored as a 5-point `rank_score` where higher = better.
6. **Min-Max normalization** — `year`, `rank_position`, `rank_score`, `query_length`,
   `query_word_count`, `location_trend_count` scaled to [0, 1] as `*_norm` columns.
7. **Comparison** — original vs normalized min/max/mean table plus a row-level preview.
8. **Verification** — no missing values, no duplicate rows, all range and consistency
   assertions pass.

Final output: **`trends_transformed.csv`**.